<a href="https://colab.research.google.com/github/rakeshma-72/LLM/blob/main/EXPERIMENT_3_Transformer_for_English_to_Hindi_Translation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers datasets sentencepiece sacrebleu torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [sacrebleu]


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sacrebleu

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [3]:
model_name = "Helsinki-NLP/opus-mt-en-hi"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
model = model.to(device)

print("Transformer model loaded successfully.")

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/812k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/1.07M [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/transformers/models/marian/tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  306MB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Transformer model loaded successfully.


In [4]:
def translate(sentence):
    # Convert English sentence into tokens
    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    # Move input to GPU/CPU
    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    # Generate Hindi translation
    with torch.no_grad():
        output_tokens = model.generate(
            **inputs,
            max_length=128,
            num_beams=4
        )

    # Convert token IDs back into Hindi text
    translation = tokenizer.decode(
        output_tokens[0],
        skip_special_tokens=True
    )

    return translation

In [5]:
english_sentences = [
    "How are you?",
    "I am a student.",
    "I am going to school.",
    "What's your name?",
    "I like reading books.",
    "She is going to school.",
    "The weather is very good today."
]

print("\nEnglish to Hindi Translation")
print("=" * 60)

translations = []

for sentence in english_sentences:
    hindi = translate(sentence)
    translations.append(hindi)

    print("English :", sentence)
    print("Hindi   :", hindi)
    print("-" * 60)


English to Hindi Translation
English : How are you?
Hindi   : आप कैसे हैं?
------------------------------------------------------------
English : I am a student.
Hindi   : मैं एक छात्र हूँ.
------------------------------------------------------------
English : I am going to school.
Hindi   : मैं स्कूल जा रहा हूँ.
------------------------------------------------------------
English : What's your name?
Hindi   : तुम्हारा नाम क्या है?
------------------------------------------------------------
English : I like reading books.
Hindi   : मैं किताबें पढ़ना पसंद है.
------------------------------------------------------------
English : She is going to school.
Hindi   : वह स्कूल जा रही है.
------------------------------------------------------------
English : The weather is very good today.
Hindi   : मौसम आज बहुत अच्छा है.
------------------------------------------------------------


In [6]:
references_1 = [
    "आप कैसे हैं?",
    "मैं एक छात्र हूँ।",
    "मैं स्कूल जा रहा हूँ।",
    "आपका नाम क्या है?",
    "मुझे किताबें पढ़ना पसंद है।",
    "वह स्कूल जा रही है।",
    "आज मौसम बहुत अच्छा है।"
]

references_2 = [
    "आप कैसे हैं?",
    "मैं एक छात्र हूँ।",
    "मैं स्कूल जा रहा हूँ।",
    "तुम्हारा नाम क्या है?",
    "मैं किताबें पढ़ना पसंद करता हूँ।",
    "वह स्कूल जा रही है।",
    "मौसम आज बहुत अच्छा है।"
]

In [7]:
bleu = sacrebleu.corpus_bleu(translations, [references_1, references_2])
print("\nBLEU Score:", bleu.score)


BLEU Score: 58.84363420261881
